# Project Track 3 - POD and Reduced-Order Learning

**Choose one variant:**

- **3A POD-rank sensitivity:** determine the smallest rank that preserves useful cavity physics.
- **3B Separate versus shared basis:** compare one POD basis per output with a joint scaled basis for `(u,v,p)`.

This notebook starts from the Week-4 POD idea but now separates three questions:

1. **Representation:** can the retained POD modes reconstruct the field if the exact coefficients are known?
2. **Coefficient learning:** can a small neural branch predict those coefficients from Reynolds number?
3. **Compactness:** how many coefficients and stored mode values are required?

Read the POD tutorial in the project guide before editing the notebook.

## Required files
`P3_POD_Study.ipynb`, `w4utils.py`, `w5_common.py`, `cavity_data.npz`


In [ ]:
import time, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import w4utils, w5_common
importlib.reload(w4utils); importlib.reload(w5_common)
assert w4utils.W4_UTILS_VERSION == "6.3", "Upload the revised w4utils.py (v6.3)."
assert w5_common.W5_COMMON_VERSION == "1.2", "Upload the revised w5_common.py (v1.2)."
data = w5_common.require_week4_files()

VARIANT = "3A"  # EDIT to 3B if approved.
TRAIN_RE = [100,150,200,225,250,350,400]
VAL_RE = 300
TEST_RE = [175,275,375]
RANKS = [1,2,3,4,5,6]

fit = w5_common.re_mask(data, TRAIN_RE)
val = w5_common.re_mask(data, [VAL_RE])
NFIELD = data["u"].shape[1] * data["u"].shape[2]
print("Training snapshots:", int(fit.sum()), "maximum centered rank:", int(fit.sum()-1))
print("Scalar grid values per field:", NFIELD, "three-field output size:", 3*NFIELD)


## 1. Build POD only from development fields

For one field, each complete training case is flattened into a snapshot vector. The mean field is removed and SVD is applied to the centered snapshot matrix. The spatial right-singular vectors are the POD modes.

POD is part of the learning pipeline. Computing modes from blind test fields is data leakage.

- `sep[name]["mean"]`: mean flattened field for `u`, `v`, or `p`.
- `sep[name]["modes"][k]`: spatial POD mode `k` for that field.
- `sep[name]["energy"][r-1]`: cumulative snapshot energy through rank `r`.
- `shared`: one POD decomposition of the scaled concatenated `(u,v,p)` vector.

A cumulative energy close to one is useful but does not guarantee preservation of weak corner vortices, wall gradients, or pressure structure.


In [ ]:
sep = w5_common.make_separate_pod(data, fit)
shared = w5_common.make_shared_pod(data, fit)
fig, ax = plt.subplots(1,4,figsize=(14,3.2))
for a,name in zip(ax[:3],["u","v","p"]):
    a.plot(np.arange(1,len(sep[name]["energy"])+1),sep[name]["energy"],"o-")
    a.set(title=f"separate {name}",xlabel="rank",ylabel="cumulative energy",ylim=(0.8,1.001)); a.grid(.3)
ax[3].plot(np.arange(1,len(shared["energy"])+1),shared["energy"],"o-")
ax[3].set(title="shared scaled basis",xlabel="rank",ylabel="cumulative energy",ylim=(0.8,1.001)); ax[3].grid(.3)
plt.tight_layout(); plt.show()


## 2. Separate reconstruction error from coefficient-prediction error

For every rank and basis family, the notebook now reports two errors:

- **Reconstruction-only error:** project the true validation field onto the retained modes and reconstruct it with the true projected coefficients. This tests the basis/rank.
- **Full ROM prediction error:** predict coefficients from `Re` with the branch MLP, then reconstruct. This includes both basis error and coefficient-learning error.

If reconstruction is accurate but the full prediction is poor, the branch model is the main limitation. If reconstruction is already poor, increasing or changing the branch network cannot repair the missing spatial basis.

The table also reports:

- `coefficient_count`: branch output dimension (`3*rank` for separate bases, `rank` for the shared basis);
- `output_reduction`: full field output size divided by coefficient count;
- `archive_compression`: a simple estimate including mean fields, modes, and stored development coefficients. With only a few cases this value can be modest even when the online output reduction is very large.


In [ ]:
def true_reconstruction(family, rank):
    """Reconstruct the validation field using exact projected coefficients."""
    if family == "separate":
        coeff = w5_common.separate_coefficients(data, sep, val, rank)[0]
        return w5_common.reconstruct_separate(data, sep, coeff, rank)
    coeff = w5_common.shared_coefficients(data, shared, val, rank)[0]
    return w5_common.reconstruct_shared(data, shared, coeff, rank)

def representation_size(family, rank, n_cases):
    """Return coefficient count, output reduction, and approximate archive compression."""
    raw = 3 * n_cases * NFIELD
    if family == "separate":
        coeff_count = 3 * rank
        stored = 3*NFIELD + 3*rank*NFIELD + 3*n_cases*rank
    else:
        coeff_count = rank
        stored = 3*NFIELD + rank*(3*NFIELD) + n_cases*rank
    return coeff_count, (3*NFIELD)/coeff_count, raw/stored

rows=[]; saved={}
families = ["separate"] if VARIANT == "3A" else ["separate","shared"]
for rank in RANKS:
    for family in families:
        if family == "separate":
            C = w5_common.separate_coefficients(data, sep, fit, rank)
            branch = w5_common.train_branch(data["Re"][fit], C, hidden=(32,32), seed=690)
            pred = w5_common.reconstruct_separate(
                data, sep, w5_common.branch_predict(branch,VAL_RE), rank)
        else:
            C = w5_common.shared_coefficients(data, shared, fit, rank)
            branch = w5_common.train_branch(data["Re"][fit], C, hidden=(32,32), seed=690)
            pred = w5_common.reconstruct_shared(
                data, shared, w5_common.branch_predict(branch,VAL_RE), rank)

        pred_report = w5_common.evaluate_prediction(data, VAL_RE, pred)
        rec = true_reconstruction(family, rank)
        rec_report = w5_common.evaluate_prediction(data, VAL_RE, rec)
        coeff_count, output_reduction, archive_compression = representation_size(
            family, rank, int(fit.sum()))

        rows.append({
            "rank": rank,
            "basis": family,
            "coefficient_count": coeff_count,
            "output_reduction": output_reduction,
            "archive_compression": archive_compression,
            "reconstruction_relative_L2_uv": rec_report["relative_L2_uv"],
            "reconstruction_relative_L2_p": rec_report["relative_L2_p"],
            **pred_report,
        })
        saved[(rank,family)] = (branch,pred,rec)

selection = pd.DataFrame(rows)
cols = ["rank","basis","coefficient_count","output_reduction","archive_compression",
        "reconstruction_relative_L2_uv","relative_L2_uv",
        "reconstruction_relative_L2_p","relative_L2_p",
        "wall_rms_error","div_l2_pred"]
display(selection[cols].round(6))


## 3. Freeze a model using a stated accuracy-compression-physics rule

Do not simply choose the largest rank. The default rule chooses the smallest rank whose validation velocity error is within 10% of the best tested error. Before finalizing the project, also verify that pressure and the selected physical checks are acceptable.

For Variant 3B, remember that the same nominal rank does not mean the same branch output dimension: separate rank `r` uses `3r` coefficients, while shared rank `r` uses `r` coefficient. Report this difference explicitly.


In [ ]:
best_uv = selection["relative_L2_uv"].min()
eligible = selection[selection["relative_L2_uv"] <= 1.10*best_uv].copy()
chosen = eligible.sort_values(["rank","relative_L2_p"]).iloc[0]
BEST_RANK = int(chosen["rank"])
BEST_BASIS = str(chosen["basis"])
print("Frozen choice:", BEST_RANK, BEST_BASIS)
print("Coefficient count:", int(chosen["coefficient_count"]))
print("Validation reconstruction uv error:", float(chosen["reconstruction_relative_L2_uv"]))
print("Validation full ROM uv error:", float(chosen["relative_L2_uv"]))


## 4. Rebuild with all permitted development cases and open blind tests

After rank and basis are frozen, recompute the POD basis using training plus validation cases, train the branch on all permitted development coefficients, and evaluate the untouched blind Reynolds numbers.


In [ ]:
dev = w5_common.re_mask(data, TRAIN_RE+[VAL_RE])
sep_all = w5_common.make_separate_pod(data, dev)
shared_all = w5_common.make_shared_pod(data, dev)
if BEST_BASIS == "separate":
    C = w5_common.separate_coefficients(data, sep_all, dev, BEST_RANK)
else:
    C = w5_common.shared_coefficients(data, shared_all, dev, BEST_RANK)
branch = w5_common.train_branch(data["Re"][dev], C, hidden=(32,32), seed=690)

rows=[]; stored={}
coeff_count = 3*BEST_RANK if BEST_BASIS == "separate" else BEST_RANK
for r in TEST_RE:
    coeff = w5_common.branch_predict(branch,r)
    pred = (w5_common.reconstruct_separate(data,sep_all,coeff,BEST_RANK)
            if BEST_BASIS == "separate" else
            w5_common.reconstruct_shared(data,shared_all,coeff,BEST_RANK))
    stored[r] = {f"{BEST_BASIS} POD rank {BEST_RANK}": pred}
    rows.append({"variant":VARIANT,"method":BEST_BASIS,"rank":BEST_RANK,
                 "coefficient_count":coeff_count,"Re":r,
                 **w5_common.evaluate_prediction(data,r,pred)})
results = pd.DataFrame(rows)
results.to_csv("P3_results.csv",index=False)
display(results)
fig = w5_common.plot_case_evidence(data,275,stored[275],"POD blind evidence")
plt.show()


## 5. Required low-rank failure comparison

Choose one deliberately low rank (usually rank 1 or 2) and compare it with the frozen selected model at the same validation or blind case. Identify which feature disappears first:

- centerline curvature;
- wall response;
- pressure gradient;
- primary-vortex position;
- weak corner or secondary structure.

Do not rely only on cumulative energy or streamline appearance.


In [ ]:
LOW_RANK = 1  # EDIT only if another deliberately low rank is better justified.
low_key = (LOW_RANK, "separate" if VARIANT=="3A" else BEST_BASIS)
if low_key in saved:
    _, low_pred, low_rec = saved[low_key]
    low_pred_report = w5_common.evaluate_prediction(data, VAL_RE, low_pred)
    low_rec_report = w5_common.evaluate_prediction(data, VAL_RE, low_rec)
    print("Low-rank reconstruction uv error:", low_rec_report["relative_L2_uv"])
    print("Low-rank full prediction uv error:", low_pred_report["relative_L2_uv"])
else:
    print("Requested low-rank/basis pair was not part of the selection study.")


## Required report evidence

- Define snapshot, mean field, POD mode, singular value, coefficient, and rank in your own words.
- Energy curves for every basis used.
- Reconstruction-only error and final prediction error reported separately.
- A table of rank, basis, coefficient count, output reduction, archive estimate, validation error, wall error, and divergence.
- For 3B, explain why equal rank is not equal coefficient capacity.
- Show which feature disappears first at low rank.
- Report why the selected rank is scientifically sufficient, not merely numerically best.

## Optional stretch extensions (advanced / prize-track only)

Complete the required project first. With instructor approval, choose at most one:

1. Compare linear interpolation of POD coefficients in Reynolds number with the neural branch under identical modes.
2. Add a matched-storage benchmark against coarse-grid bilinear reconstruction or per-field truncated SVD.
3. Define a structure-aware rank rule that includes a weak-vortex, wall, centerline, or pressure-gradient criterion rather than energy alone.
4. Add more physical cases and examine how true archive compression changes with dataset size.
